# A/B Testing & Marketing Strategy Analysis

## Phase 1: Data Validation

Objective:
Validate data quality before performing any statistical testing.

Checks Included:

1. Dataset shape verification
2. Column verification
3. Group size validation
4. Missing value analysis
5. Duplicate record analysis
6. Sample Ratio Mismatch (SRM) validation
7. Group comparability checks

Business Rule:

No hypothesis test will be trusted until data quality and randomization integrity are verified.

In [1]:
import pandas as pd
import numpy as np

from scipy.stats import chisquare
from scipy.stats import chi2_contingency

import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", None)

print("Libraries Loaded Successfully")

Libraries Loaded Successfully


In [2]:
df = pd.read_csv("../data/marketing_AB.csv")

print("Dataset Loaded Successfully")

Dataset Loaded Successfully


In [3]:
print("Rows :", df.shape[0])
print("Columns :", df.shape[1])

print("\nColumn Names:\n")
print(df.columns.tolist())

Rows : 588101
Columns : 7

Column Names:

['Unnamed: 0', 'user id', 'test group', 'converted', 'total ads', 'most ads day', 'most ads hour']


In [4]:
df.head()

,Unnamed: 0,user id,test group,converted,total ads,most ads day,most ads hour
0,0,1069124,ad,False,130,Monday,20
1,1,1119715,ad,False,93,Tuesday,22
2,2,1144181,ad,False,21,Tuesday,18
3,3,1435133,ad,False,355,Tuesday,10
4,4,1015700,ad,False,276,Friday,14


In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 588101 entries, 0 to 588100
Data columns (total 7 columns):
 #   Column         Non-Null Count   Dtype 
---  ------         --------------   ----- 
 0   Unnamed: 0     588101 non-null  int64 
 1   user id        588101 non-null  int64 
 2   test group     588101 non-null  object
 3   converted      588101 non-null  bool  
 4   total ads      588101 non-null  int64 
 5   most ads day   588101 non-null  object
 6   most ads hour  588101 non-null  int64 
dtypes: bool(1), int64(4), object(2)
memory usage: 27.5+ MB


In [6]:
df.describe(include="all")

,Unnamed: 0,user id,test group,converted,total ads,most ads day,most ads hour
count,588101.000000,5.881010e+05,588101,588101,588101.000000,588101,588101.000000
unique,NaN,NaN,2,2,NaN,7,NaN
top,NaN,NaN,ad,False,NaN,Friday,NaN
freq,NaN,NaN,564577,573258,NaN,92608,NaN
mean,294050.000000,1.310692e+06,NaN,NaN,24.820876,NaN,14.469061
std,169770.279668,2.022260e+05,NaN,NaN,43.715181,NaN,4.834634
min,0.000000,9.000000e+05,NaN,NaN,1.000000,NaN,0.000000
25%,147025.000000,1.143190e+06,NaN,NaN,4.000000,NaN,11.000000
50%,294050.000000,1.313725e+06,NaN,NaN,13.000000,NaN,14.000000
75%,441075.000000,1.484088e+06,NaN,NaN,27.000000,NaN,18.000000


In [7]:
df.columns = (
    df.columns
      .str.strip()
      .str.lower()
      .str.replace(" ", "_")
)

df.columns

Index(['unnamed:_0', 'user_id', 'test_group', 'converted', 'total_ads',
       'most_ads_day', 'most_ads_hour'],
      dtype='object')

In [8]:
df.head()

,unnamed:_0,user_id,test_group,converted,total_ads,most_ads_day,most_ads_hour
0,0,1069124,ad,False,130,Monday,20
1,1,1119715,ad,False,93,Tuesday,22
2,2,1144181,ad,False,21,Tuesday,18
3,3,1435133,ad,False,355,Tuesday,10
4,4,1015700,ad,False,276,Friday,14


In [9]:
df = df.drop(columns=["unnamed:_0"])

In [10]:
df.columns

Index(['user_id', 'test_group', 'converted', 'total_ads', 'most_ads_day',
       'most_ads_hour'],
      dtype='object')

In [11]:
missing = df.isnull().sum()

missing

user_id          0
test_group       0
converted        0
total_ads        0
most_ads_day     0
most_ads_hour    0
dtype: int64

In [12]:
missing_percentage = (
    df.isnull().mean()*100
).round(2)

missing_percentage

user_id          0.0
test_group       0.0
converted        0.0
total_ads        0.0
most_ads_day     0.0
most_ads_hour    0.0
dtype: float64

### Missing Value Assessment

No missing values were detected in critical experiment variables.

The dataset is suitable for statistical analysis without imputation.

In [14]:
df["user_id"].nunique()

588101

In [15]:
duplicates = (
    df.duplicated(subset="user_id")
      .sum()
)

print("Duplicate Users:", duplicates)

Duplicate Users: 0


In [16]:
df.shape[0]

588101

In [17]:
group_counts = (
    df["test_group"]
      .value_counts()
)

group_counts

test_group
ad     564577
psa     23524
Name: count, dtype: int64

In [18]:
(
    df["test_group"]
      .value_counts(normalize=True)
      *100
).round(2)

test_group
ad     96.0
psa     4.0
Name: proportion, dtype: float64

In [19]:
df["test_group"].unique()

array(['ad', 'psa'], dtype=object)

In [20]:
validation_summary = pd.DataFrame({
    "Metric":[
        "Total Rows",
        "Unique Users",
        "Duplicate Users",
        "Missing Values"
    ],
    "Value":[
        len(df),
        df["user_id"].nunique(),
        duplicates,
        df.isnull().sum().sum()
    ]
})

validation_summary

,Metric,Value
0,Total Rows,588101
1,Unique Users,588101
2,Duplicate Users,0
3,Missing Values,0


## Data Quality Assessment

The dataset contains 588,101 observations and 588,101 unique users.

No duplicate users were detected.

No missing values were found in any experiment variable.

Therefore, the dataset satisfies the minimum data quality requirements for statistical analysis and hypothesis testing.

In [21]:
df.head()

,user_id,test_group,converted,total_ads,most_ads_day,most_ads_hour
0,1069124,ad,False,130,Monday,20
1,1119715,ad,False,93,Tuesday,22
2,1144181,ad,False,21,Tuesday,18
3,1435133,ad,False,355,Tuesday,10
4,1015700,ad,False,276,Friday,14


In [22]:
df.sample(5)

,user_id,test_group,converted,total_ads,most_ads_day,most_ads_hour
102028,1280922,ad,False,5,Wednesday,14
562993,1226845,ad,False,5,Friday,10
421132,1028304,ad,False,4,Thursday,18
50954,1448270,ad,False,81,Sunday,16
272666,1413872,ad,False,16,Wednesday,11


# Sample Ratio Mismatch (SRM) Validation

Before analyzing conversion outcomes, we must verify that user assignment between experiment groups is consistent with the intended allocation.

Sample Ratio Mismatch (SRM) occurs when observed group sizes differ significantly from expected group sizes.

SRM can indicate:

- Randomization failures
- Logging issues
- Traffic allocation bugs
- Data collection errors

If SRM is detected, experiment results may be invalid regardless of statistical significance.

Therefore, SRM validation is performed before any hypothesis testing.

In [23]:
observed = (
    df["test_group"]
      .value_counts()
      .sort_index()
)

observed

test_group
ad     564577
psa     23524
Name: count, dtype: int64

In [24]:
(
    df["test_group"]
      .value_counts(normalize=True)
      .sort_index()
      *100
).round(2)

test_group
ad     96.0
psa     4.0
Name: proportion, dtype: float64

In [25]:
total_users = len(df)

expected_ad = total_users * 0.96
expected_psa = total_users * 0.04

expected = [
    expected_ad,
    expected_psa
]

expected

[564576.96, 23524.04]

In [26]:
from scipy.stats import chisquare

chi2_stat, p_value = chisquare(
    f_obs=observed.values,
    f_exp=expected
)

print("Chi-square Statistic:", round(chi2_stat,4))
print("P-value:", round(p_value,4))

Chi-square Statistic: 0.0
P-value: 0.9998


In [27]:
alpha = 0.05

if p_value < alpha:
    print("Potential SRM Detected")
else:
    print("No Evidence of SRM")

No Evidence of SRM


## SRM Interpretation

The chi-square goodness-of-fit test compares observed traffic allocation against the expected allocation.

If the p-value exceeds 0.05, we fail to detect evidence of Sample Ratio Mismatch (SRM), suggesting that traffic assignment appears consistent with the intended experiment design.

Had SRM been detected, the experiment results would be considered unreliable because a randomization failure may bias conversion estimates. In such a scenario, the experiment would require investigation before any business conclusions could be trusted.

In [28]:
df.groupby("test_group")["total_ads"].describe()

,count,mean,std,min,25%,50%,75%,max
test_group,,,,,,,,
ad,564577.0,24.823365,43.750456,1.0,4.0,13.0,27.0,2065.0
psa,23524.0,24.761138,42.860720,1.0,4.0,12.0,26.0,907.0


In [29]:
df.groupby("test_group")["total_ads"].agg(
    ["mean","median","std","min","max"]
)

,mean,median,std,min,max
test_group,,,,,
ad,24.823365,13.0,43.750456,1,2065
psa,24.761138,12.0,42.860720,1,907


## Group Comparability Conclusion

The advertising (ad) and PSA groups exhibit highly similar ad exposure distributions.

Mean ad exposure differs by less than 0.1 advertisements per user (24.82 vs 24.76), while median exposure is also nearly identical (13 vs 12).

This suggests that any observed differences in conversion are unlikely to be driven by systematic differences in ad exposure volume.

Therefore, the groups appear sufficiently comparable for conversion-rate analysis.

# Final Validation Summary

### Data Quality

- No missing values detected
- No duplicate users detected
- 588,101 unique users verified

### Randomization Integrity

- No evidence of Sample Ratio Mismatch (SRM)
- Observed allocation matches expected experiment allocation

### Group Comparability

- Ad exposure distributions are highly similar across groups
- No major imbalance detected

### Conclusion

The dataset passes all major validation checks and is suitable for conversion-rate analysis and statistical hypothesis testing.